## Example 1: Basic nest_asyncio patching and nested event loop support

In [ ]:
import nest_asyncio
import asyncio

print("Example 1: Basic nest_asyncio patching and nested event loop support")

try:
    print(f"nest_asyncio version: {getattr(nest_asyncio, '__version__', 'unknown')}")
    
    # Apply the patch to allow nested event loops
    nest_asyncio.apply()
    print("✓ nest_asyncio.apply() executed successfully")
    
    # Test nested async function
    async def simple_coroutine():
        await asyncio.sleep(0.1)
        return "Hello from nested async!"
    
    # This works in Jupyter because nest_asyncio allows nested loops
    result = await simple_coroutine()
    print(f"Async result: {result}")
    
    # Show current event loop information
    loop = asyncio.get_running_loop()
    print(f"Current loop: {type(loop).__name__}")
    print(f"Loop is running: {loop.is_running()}")
    
    # Test multiple nested calls
    async def nested_operation(name, delay):
        print(f"Starting {name}")
        await asyncio.sleep(delay)
        print(f"Completed {name}")
        return f"Result from {name}"
    
    results = await asyncio.gather(
        nested_operation("Task 1", 0.1),
        nested_operation("Task 2", 0.05),
        nested_operation("Task 3", 0.15)
    )
    print(f"Gathered results: {results}")
    
    print("nest_asyncio basic functionality completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating nest_asyncio concepts:")
    concepts = [
        "Event loop patching: Allows running asyncio.run() in already running loops",
        "Jupyter compatibility: Enables async/await in Jupyter notebooks", 
        "Nested execution: Supports nested async function calls"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 1 finished")

## Example 2: Synchronous code calling async functions

In [ ]:
import nest_asyncio
import asyncio
import time

print("Example 2: Synchronous code calling async functions")

try:
    # Ensure nest_asyncio is applied
    nest_asyncio.apply()
    
    # Define async functions to be called from sync code
    async def fetch_data(url_id, delay):
        print(f"Fetching data from URL {url_id}")
        await asyncio.sleep(delay)  # Simulate network delay
        return f"Data from URL {url_id}"
    
    async def process_data(data):
        print(f"Processing: {data}")
        await asyncio.sleep(0.1)  # Simulate processing time
        return f"Processed {data}"
    
    # Function that can be called from synchronous code
    def sync_function_calling_async():
        print("Synchronous function starting...")
        
        # This would normally fail without nest_asyncio
        loop = asyncio.get_event_loop()
        
        # Call async function from sync context
        result = loop.run_until_complete(fetch_data("API-1", 0.2))
        print(f"Sync got result: {result}")
        
        # Chain multiple async calls
        processed = loop.run_until_complete(process_data(result))
        print(f"Sync got processed result: {processed}")
        
        return processed
    
    # This works because nest_asyncio allows nested event loops
    start_time = time.time()
    final_result = sync_function_calling_async()
    duration = time.time() - start_time
    
    print(f"Final result: {final_result}")
    print(f"Execution time: {duration:.3f} seconds")
    
    # Test with multiple sync functions
    def sync_batch_processing():
        results = []
        for i in range(3):
            loop = asyncio.get_event_loop()
            data = loop.run_until_complete(fetch_data(f"Batch-{i}", 0.1))
            processed = loop.run_until_complete(process_data(data))
            results.append(processed)
        return results
    
    batch_results = sync_batch_processing()
    print(f"Batch processing results: {batch_results}")
    
    print("Synchronous-async integration completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating sync-async concepts:")
    concepts = [
        "run_until_complete: Execute async functions from sync context",
        "Nested loops: Allow asyncio.run() in running event loops",
        "Legacy integration: Support older sync codebases with async libraries"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 2 finished")

## Example 3: Mixed async/await and asyncio.run() patterns

In [ ]:
import nest_asyncio
import asyncio
import concurrent.futures

print("Example 3: Mixed async/await and asyncio.run() patterns")

try:
    # Apply nest_asyncio patching
    nest_asyncio.apply()
    
    # Async function that itself uses asyncio.run()
    async def outer_async_function():
        print("Outer async function started")
        
        # Define an inner async function
        async def inner_async_task():
            await asyncio.sleep(0.1)
            return "Inner task completed"
        
        # This would normally cause "RuntimeError: asyncio.run() cannot be called from a running event loop"
        # But with nest_asyncio, it works
        result = asyncio.run(inner_async_task())
        print(f"Inner result: {result}")
        
        return f"Outer function got: {result}"
    
    # Call the outer function - this works in Jupyter with nest_asyncio
    outer_result = await outer_async_function()
    print(f"Final result: {outer_result}")
    
    # Demonstrate complex nested patterns
    async def complex_nested_operations():
        print("Starting complex nested operations...")
        
        # Mix of different async patterns
        tasks = []
        
        # Direct await
        await asyncio.sleep(0.05)
        print("Direct await completed")
        
        # asyncio.run() within async function
        def sync_wrapper():
            async def inner():
                await asyncio.sleep(0.05)
                return "From sync wrapper"
            return asyncio.run(inner())
        
        sync_result = sync_wrapper()
        print(f"Sync wrapper result: {sync_result}")
        
        # Concurrent execution
        async def concurrent_task(task_id):
            await asyncio.sleep(0.1)
            return f"Task {task_id} done"
        
        concurrent_results = await asyncio.gather(
            concurrent_task(1),
            concurrent_task(2),
            concurrent_task(3)
        )
        print(f"Concurrent results: {concurrent_results}")
        
        return "Complex operations completed"
    
    complex_result = await complex_nested_operations()
    print(f"Complex operations result: {complex_result}")
    
    # Test thread pool execution with async
    async def async_with_threads():
        print("Testing async with thread pool...")
        
        def cpu_bound_task(n):
            return sum(i * i for i in range(n))
        
        with concurrent.futures.ThreadPoolExecutor() as executor:
            # Run CPU-bound tasks in threads
            loop = asyncio.get_running_loop()
            results = await asyncio.gather(
                loop.run_in_executor(executor, cpu_bound_task, 1000),
                loop.run_in_executor(executor, cpu_bound_task, 2000),
                loop.run_in_executor(executor, cpu_bound_task, 3000)
            )
        
        print(f"Thread pool results: {results}")
        return results
    
    thread_results = await async_with_threads()
    
    print("Mixed async/await patterns completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating nested pattern concepts:")
    concepts = [
        "Nested asyncio.run(): Allow asyncio.run() within async functions",
        "Complex workflows: Support mixed sync/async execution patterns",
        "Thread integration: Combine async with thread pool execution"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 3 finished")

## Example 4: Integration with third-party async libraries

In [ ]:
import nest_asyncio
import asyncio
import json

print("Example 4: Integration with third-party async libraries")

try:
    # Apply nest_asyncio for compatibility
    nest_asyncio.apply()
    
    # Simulate third-party async library that uses asyncio.run()
    class AsyncDataProcessor:
        """Simulated third-party library that processes data asynchronously"""
        
        @staticmethod
        async def fetch_and_process(source_id, processing_time=0.1):
            """Simulate fetching and processing data"""
            print(f"Fetching data from source {source_id}")
            await asyncio.sleep(processing_time)
            
            # Simulate data processing
            data = {"source": source_id, "timestamp": processing_time * 1000, "status": "processed"}
            print(f"Data processed for source {source_id}")
            return data
        
        @classmethod
        def process_sync(cls, source_id, processing_time=0.1):
            """Synchronous wrapper that would normally fail in Jupyter"""
            # This would normally raise RuntimeError without nest_asyncio
            return asyncio.run(cls.fetch_and_process(source_id, processing_time))
    
    # Test the processor in a Jupyter-like environment
    print("Testing async data processor...")
    
    # Direct async call
    async_result = await AsyncDataProcessor.fetch_and_process("async-source-1", 0.15)
    print(f"Async result: {json.dumps(async_result, indent=2)}")
    
    # Sync wrapper call (would fail without nest_asyncio)
    sync_result = AsyncDataProcessor.process_sync("sync-source-1", 0.12)
    print(f"Sync wrapper result: {json.dumps(sync_result, indent=2)}")
    
    # Simulate library that mixes both approaches
    class HybridAsyncLibrary:
        """Library that uses both async/await and asyncio.run() patterns"""
        
        def __init__(self):
            self.cache = {}
        
        async def async_operation(self, key, data):
            """Pure async operation"""
            await asyncio.sleep(0.05)
            self.cache[key] = data
            return f"Async stored: {key}"
        
        def sync_batch_operation(self, batch_data):
            """Sync operation that calls async methods internally"""
            results = []
            for key, data in batch_data.items():
                # This works because of nest_asyncio
                result = asyncio.run(self.async_operation(key, data))
                results.append(result)
            return results
        
        async def hybrid_operation(self, batch_data):
            """Async operation that calls sync methods that use asyncio.run()"""
            print("Starting hybrid operation...")
            
            # Call the sync method that internally uses asyncio.run()
            sync_results = self.sync_batch_operation(batch_data)
            
            # Additional async processing
            await asyncio.sleep(0.1)
            
            return {
                "sync_results": sync_results,
                "cache_size": len(self.cache),
                "final_status": "hybrid_complete"
            }
    
    # Test hybrid library
    hybrid_lib = HybridAsyncLibrary()
    test_data = {
        "user_1": {"name": "Alice", "score": 95},
        "user_2": {"name": "Bob", "score": 87},
        "user_3": {"name": "Charlie", "score": 92}
    }
    
    hybrid_result = await hybrid_lib.hybrid_operation(test_data)
    print(f"Hybrid operation result: {json.dumps(hybrid_result, indent=2)}")
    
    # Test concurrent operations with third-party libraries
    async def concurrent_third_party_calls():
        """Test concurrent calls to libraries that use asyncio.run()"""
        print("Testing concurrent third-party calls...")
        
        # Create multiple processors
        processors = [AsyncDataProcessor() for _ in range(3)]
        
        # Concurrent sync calls (normally would fail)
        tasks = []
        for i, processor in enumerate(processors):
            # These sync calls work because of nest_asyncio
            task = asyncio.create_task(
                asyncio.to_thread(processor.process_sync, f"concurrent-{i}", 0.1)
            )
            tasks.append(task)
        
        concurrent_results = await asyncio.gather(*tasks)
        return concurrent_results
    
    concurrent_results = await concurrent_third_party_calls()
    print(f"Concurrent third-party results: {len(concurrent_results)} items")
    for i, result in enumerate(concurrent_results):
        print(f"  Result {i+1}: {result['source']} -> {result['status']}")
    
    print("Third-party async library integration completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating third-party integration concepts:")
    concepts = [
        "Library compatibility: Enable libraries that use asyncio.run() in Jupyter",
        "Hybrid patterns: Support mixed async/sync third-party code",
        "Concurrent integration: Allow parallel execution of nested async operations"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 4 finished")

## Example 5: Production deployment patterns and best practices

In [ ]:
import nest_asyncio
import asyncio
import sys
import platform

print("Example 5: Production deployment patterns and best practices")

try:
    # Check environment and apply nest_asyncio conditionally
    def setup_nest_asyncio():
        """Setup nest_asyncio with environment detection"""
        print(f"Python version: {sys.version}")
        print(f"Platform: {platform.system()}")
        print(f"asyncio policy: {type(asyncio.get_event_loop_policy()).__name__}")
        
        # Check if we're in an environment that needs nest_asyncio
        try:
            # This will fail if we're already in a running loop
            loop = asyncio.get_running_loop()
            print("Running in event loop - applying nest_asyncio")
            nest_asyncio.apply()
            return True
        except RuntimeError:
            print("No running event loop detected")
            return False
    
    nested_applied = setup_nest_asyncio()
    
    # Production-ready async application structure
    class AsyncApplication:
        """Production async application with proper lifecycle management"""
        
        def __init__(self, name="AsyncApp"):
            self.name = name
            self.running = False
            self.tasks = []
            
        async def startup(self):
            """Application startup with proper initialization"""
            print(f"Starting {self.name}...")
            self.running = True
            
            # Initialize async resources
            await asyncio.sleep(0.05)  # Simulate startup time
            print(f"✓ {self.name} startup complete")
            
        async def shutdown(self):
            """Graceful shutdown with cleanup"""
            print(f"Shutting down {self.name}...")
            self.running = False
            
            # Cancel running tasks
            for task in self.tasks:
                if not task.done():
                    task.cancel()
                    try:
                        await task
                    except asyncio.CancelledError:
                        pass
            
            self.tasks.clear()
            print(f"✓ {self.name} shutdown complete")
            
        async def background_worker(self, worker_id, work_time=0.1):
            """Background worker task"""
            try:
                while self.running:
                    print(f"Worker {worker_id} processing...")
                    await asyncio.sleep(work_time)
                    
                    # Simulate work that might use asyncio.run() (with nest_asyncio)
                    if nested_applied:
                        def sync_work():
                            async def inner_work():
                                await asyncio.sleep(0.02)
                                return f"Worker {worker_id} inner work complete"
                            return asyncio.run(inner_work())
                        
                        inner_result = sync_work()
                        print(f"  {inner_result}")
                    
                    await asyncio.sleep(0.2)  # Worker cycle time
                    
            except asyncio.CancelledError:
                print(f"Worker {worker_id} cancelled")
                raise
                
        async def run_application(self, duration=1.0):
            """Run application with proper lifecycle management"""
            try:
                await self.startup()
                
                # Start background workers
                worker_tasks = [
                    asyncio.create_task(self.background_worker(f"W{i}", 0.1))
                    for i in range(2)
                ]
                self.tasks.extend(worker_tasks)
                
                # Run application for specified duration
                await asyncio.sleep(duration)
                
            finally:
                await self.shutdown()
    
    # Test application
    app = AsyncApplication("TestApp")
    await app.run_application(duration=0.5)
    
    # Production deployment considerations
    def deployment_checklist():
        """Production deployment checklist for nest_asyncio applications"""
        checklist = {
            "Environment Detection": "✓ Detect if nest_asyncio is needed",
            "Conditional Application": "✓ Apply only when necessary",
            "Error Handling": "✓ Handle RuntimeError gracefully",
            "Performance Impact": "✓ Monitor performance overhead",
            "Testing": "✓ Test in both Jupyter and production environments",
            "Documentation": "✓ Document nest_asyncio usage for team",
        }
        
        print("\nProduction Deployment Checklist:")
        for item, status in checklist.items():
            print(f"  {status} {item}")
        
        return checklist
    
    checklist = deployment_checklist()
    
    # Best practices demonstration
    print("\nBest Practices:")
    best_practices = [
        "Only apply nest_asyncio when actually needed (e.g., in Jupyter)",
        "Use try/except blocks around asyncio.get_running_loop() checks", 
        "Test applications both with and without nest_asyncio",
        "Monitor for performance impacts in production",
        "Document dependencies on nest_asyncio for maintenance",
        "Consider alternative architectures that don't require nested loops"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"  {i}. {practice}")
    
    # Environment-specific configuration
    config = {
        "jupyter_env": nested_applied,
        "async_compatible": True,
        "production_ready": True,
        "performance_overhead": "minimal"
    }
    
    print(f"\nEnvironment Configuration: {config}")
    print("Production deployment patterns completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating production concepts:")
    concepts = [
        "Environment detection: Check if nest_asyncio is needed",
        "Conditional application: Apply patching only when necessary",
        "Lifecycle management: Proper startup/shutdown with async applications",
        "Performance monitoring: Track overhead in production environments"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 5 finished")